In [0]:
import pyspark.sql.functions as F

In [0]:
source_path = '/Volumes/finguard/source/fraud_watchlist/source_data/'

In [0]:
# Check is there files in the volume path
dbutils.fs.ls(source_path)

In [0]:
# Create a Dataframe
# 1. format("cloudFiles") means we are using Autoloader
# 2. For Autoloader only readStream is supported, so we can use readStream for batch ingestion saying the triggerType(availableNow = True)

# Define the schema explicitly based on the JSON structure
from pyspark.sql.types import StructType, StructField, StringType

schema = StructType([
    StructField("watchlist_id", StringType(), True),
    StructField("watchlist_type", StringType(), True),
    StructField("entity_id", StringType(), True),
    StructField("risk_level", StringType(), True),
    StructField("action", StringType(), True),
    StructField("reason_code", StringType(), True),
    StructField("reason_description", StringType(), True),
    StructField("status", StringType(), True),
    StructField("effective_from", StringType(), True),
    StructField("reported_by", StringType(), True),
    StructField("reported_source", StringType(), True),
    StructField("country", StringType(), True),
    StructField("city", StringType(), True),
    StructField("_rescued_data", StringType(), True)
])

input_stream = (
    spark.readStream.format("cloudFiles")
                    .option("cloudFiles.format","json")
                    .option("cloudFiles.schemaLocation","/Volumes/finguard/source/fraud_watchlist/schema/")
                    # Previously we used option("cloudFiles.inferColumnTypes", "true"), but it is not working, so the schema is created manually, but we need to check this
                    .schema(schema)
                    .load(source_path)
)

In [0]:
# By this we can get additional file related properties also
transformed_df = input_stream.select(
    "*",
    F.col("_metadata.file_path").alias("file_path"),
    F.current_timestamp().alias("ingest_timestamp")
)

In [0]:
streaming_query = (transformed_df.writeStream.format("delta").outputMode("append")
                                # Create a new directory in the volume location
                                .option("checkpointLocation","/Volumes/finguard/source/fraud_watchlist/checkpoint/")
                                .trigger(availableNow = True)
                                .toTable("finguard.bronze.fraud_watchlist_batch_test")
                                )

In [0]:
%sql
select * from finguard.gold.fraud_card_alert

In [0]:
%sql
select * from finguard.bronze.fraud_watchlist_batch_test